#0. INSTALL & SETUP LIBRARY

In [ ]:
!pip install nltk Sastrawi pandas numpy tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.8 MB/s eta 0:00:00


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import re
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# 1. MENYIAPKAN DOKUMEN

In [ ]:
# 1. Menyiapkan Dokumen
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# a. Baca isi file txt
with open(filename, "r", encoding="utf-8") as f:
    original_text = f.read()

# Hitung jumlah baris
with open(filename, "r", encoding="utf-8") as f:
    lines = f.readlines()

print(">>> File dibaca")
print("Panjang karakter :", len(original_text))
print("Jumlah baris     :", len(lines))


Saving ETA_PedomanAkademik.txt to ETA_PedomanAkademik.txt
>>> File dibaca
Panjang karakter : 185198
Jumlah baris     : 1414


#2. MEMBACA FILE ETALASE (LAYER A)

In [ ]:
path_manusia = "ETA_PedomanAkademik.txt"
path_mesin = "MES_PedomanAkademik.txt" # Nama output file mesin

with open(path_manusia, "r", encoding="utf-8") as f:
    # Kita baca per baris karena file Etalase sudah disegmentasi dengan benar
    sentences = [line.strip() for line in f.readlines()]

print(f">>> Jumlah kalimat dari Etalase: {len(sentences)}")

>>> Jumlah kalimat dari Etalase: 1414


# 3. PREPROCESSING TEKS (MEMBENTUK LAYER B)

In [ ]:
# Case Folding,  Pembersihan karakter non-alfanumerik, Tokenization, Stopword Removal, Stemming.
stopwords_id = set(stopwords.words('indonesian'))
factory = StemmerFactory()
stemmer = factory.create_stemmer()

processed_sentences = []

# Jumlah kalimat yang ingin ditampilkan sebagai contoh
prepro_tampil = 3

print(">>> Sedang memproses (ini mungkin memakan waktu)...")

for i, s in enumerate(sentences):
    # a. Case folding dan pembersihan ringan
    # Mengubah huruf menjadi lowercase dan menghapus tanda baca
    s_lower = s.lower()
    s_clean = re.sub(r'[^a-z0-9\s]', '', s_lower)

    # b. Tokenisasi kata
    tokens_awal = word_tokenize(s_clean)

    # c. Stopword removal
    # Menghapus kata umum yang tidak berkontribusi pada makna
    tokens_stop = [t for t in tokens_awal if t not in stopwords_id]

    # d. Stemming
    # Mengubah kata ke bentuk dasarnya
    tokens_stem = [stemmer.stem(t) for t in tokens_stop]

    # Menggabungkan kembali token menjadi satu string
    hasil_akhir = ' '.join(tokens_stem)

    # PENTING:
    # Kalimat tetap disimpan meskipun hasil preprocessing kosong
    # untuk menjaga kesesuaian indeks antara teks asli dan teks mesin
    processed_sentences.append(hasil_akhir)

      # TAMPILKAN Hasil PREPROCESSING
    if i < prepro_tampil:
        print(f"Kalimat ke-{i+1}")
        print("Teks asli        :", s)
        print("a. Case folding  :", s_lower)
        print("a. Cleaning     :", s_clean)
        print("b. Tokenisasi   :", tokens_awal)
        print("c. Stopword     :", tokens_stop)
        print("d. Stemming     :", tokens_stem)
        print("Hasil akhir     :", hasil_akhir)
        print("-" * 60)

print(f">>> Selesai memproses {len(processed_sentences)} baris.")


>>> Sedang memproses (ini mungkin memakan waktu)...
Kalimat ke-1
Teks asli        : Pada tahun lima puluhan, setelah selesai revolusi, fisik, pertumbuhan Sekolah Menengah Pertama (SMP) dan Sekolah Menengah Atas (SMA) demikian pesatnya, sehingga kebutuhan akan tenaga kependidikan (guru) yang berkualitas saat itu dapat dipenuhi.
a. Case folding  : pada tahun lima puluhan, setelah selesai revolusi, fisik, pertumbuhan sekolah menengah pertama (smp) dan sekolah menengah atas (sma) demikian pesatnya, sehingga kebutuhan akan tenaga kependidikan (guru) yang berkualitas saat itu dapat dipenuhi.
a. Cleaning     : pada tahun lima puluhan setelah selesai revolusi fisik pertumbuhan sekolah menengah pertama smp dan sekolah menengah atas sma demikian pesatnya sehingga kebutuhan akan tenaga kependidikan guru yang berkualitas saat itu dapat dipenuhi
b. Tokenisasi   : ['pada', 'tahun', 'lima', 'puluhan', 'setelah', 'selesai', 'revolusi', 'fisik', 'pertumbuhan', 'sekolah', 'menengah', 'pertama', 'smp', '

# 4. MENYIMPAN FILE HASIL PREPROCESSING (LAYER B)

In [ ]:
with open(path_mesin, "w", encoding="utf-8") as f:
    for s in processed_sentences:
        f.write(s + "\n")

print(f">>> File hasil preprocessing disimpan: {path_mesin}")

>>> File hasil preprocessing disimpan: MES_PedomanAkademik.txt


# 5. PENGECEKAN PARALELISME DATA

In [ ]:
if len(sentences) == len(processed_sentences):
    print("\n✅ SUKSES: File PARALEL! Jumlah baris sama persis.")
    print(f"Manusia: {len(sentences)} baris")
    print(f"Mesin  : {len(processed_sentences)} baris")
else:
    print("\n❌ ERROR: Tidak paralel (Sangat aneh jika ini terjadi dengan skrip ini).")


✅ SUKSES: File PARALEL! Jumlah baris sama persis.
Manusia: 1414 baris
Mesin  : 1414 baris
